# 3D Parallelism: Combining TP, PP, and DP

## Overview

This tutorial explains how to combine tensor, pipeline, and data parallelism for training very large models.

### Topics Covered
- 3D parallelism architecture
- Process group configuration
- Communication patterns
- Optimal parallelism selection

## 1. 3D Parallelism Architecture

```
Total GPUs = TP × PP × DP

Example: 64 GPUs = 8 (TP) × 4 (PP) × 2 (DP)

┌─────────────────────────────────────────────────────────────┐
│                     Data Parallel Replica 0                 │
│  ┌─────────────────────────────────────────────────────┐   │
│  │ Pipeline Stage 0    Stage 1    Stage 2    Stage 3   │   │
│  │ ┌───────────┐     ┌───────┐   ┌───────┐  ┌───────┐ │   │
│  │ │TP: 8 GPUs │ --> │8 GPUs │-->│8 GPUs │->│8 GPUs │ │   │
│  │ │Layer 0-5  │     │L 6-11 │   │L 12-17│  │L 18-23│ │   │
│  │ └───────────┘     └───────┘   └───────┘  └───────┘ │   │
│  └─────────────────────────────────────────────────────┘   │
├─────────────────────────────────────────────────────────────┤
│                     Data Parallel Replica 1                 │
│  ┌─────────────────────────────────────────────────────┐   │
│  │ (Same structure, different data)                     │   │
│  └─────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
from dataclasses import dataclass

@dataclass
class ParallelConfig:
    """3D parallelism configuration."""
    tensor_parallel: int = 1   # TP
    pipeline_parallel: int = 1  # PP
    data_parallel: int = 1      # DP
    
    @property
    def world_size(self):
        return self.tensor_parallel * self.pipeline_parallel * self.data_parallel
    
    def get_rank_info(self, global_rank):
        """Get TP, PP, DP ranks from global rank."""
        tp = self.tensor_parallel
        pp = self.pipeline_parallel
        
        tp_rank = global_rank % tp
        pp_rank = (global_rank // tp) % pp
        dp_rank = global_rank // (tp * pp)
        
        return {'tp_rank': tp_rank, 'pp_rank': pp_rank, 'dp_rank': dp_rank}

# Example
config = ParallelConfig(tensor_parallel=8, pipeline_parallel=4, data_parallel=2)
print(f"Total GPUs: {config.world_size}")
print(f"Rank 0: {config.get_rank_info(0)}")
print(f"Rank 32: {config.get_rank_info(32)}")

## 2. Communication Patterns

| Parallelism | Communication | Scope | Frequency |
|-------------|---------------|-------|----------|
| Tensor (TP) | AllReduce | Within TP group | Per layer |
| Pipeline (PP) | P2P Send/Recv | Adjacent stages | Per micro-batch |
| Data (DP) | AllReduce | Across DP replicas | Per step |

## 3. Optimal Configuration Selection

### Guidelines

```
1. Tensor Parallel (TP):
   - Use within a node (high bandwidth NVLink)
   - Typical: 2, 4, or 8 (GPUs per node)
   - Increases with model hidden size

2. Pipeline Parallel (PP):
   - Use across nodes (lower bandwidth)
   - Typical: 2, 4, 8
   - Increases with model depth (layers)

3. Data Parallel (DP):
   - Use remaining GPUs
   - DP = Total_GPUs / (TP × PP)
   - Increases throughput
```

In [ ]:
def recommend_3d_config(num_gpus, model_params_b, gpus_per_node=8):
    """Recommend 3D parallelism configuration."""
    
    # Heuristics based on model size
    if model_params_b < 10:
        tp = min(2, gpus_per_node)
        pp = 1
    elif model_params_b < 50:
        tp = min(4, gpus_per_node)
        pp = 2
    elif model_params_b < 200:
        tp = gpus_per_node
        pp = 4
    else:
        tp = gpus_per_node
        pp = 8
    
    dp = num_gpus // (tp * pp)
    
    print(f"Model: {model_params_b}B params, {num_gpus} GPUs")
    print(f"Recommended: TP={tp}, PP={pp}, DP={dp}")
    print(f"Total: {tp * pp * dp} GPUs")
    
    return tp, pp, dp

recommend_3d_config(256, 175)  # GPT-3 scale

## 4. Summary

### Key Takeaways

1. **TP within node**: High bandwidth, low latency
2. **PP across nodes**: Moderate bandwidth requirement
3. **DP for scaling**: Increases throughput linearly
4. **Balance**: Minimize communication, maximize compute